# Quickstart: audit a spatial cohort

This example uses the package’s deterministic synthetic composition fixture. Install `spatial-audit` in the active Python 3.13 environment. Coordinates in this example are micrometres. The notebook reads no external data and writes its example files to a temporary directory.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import anndata as ad
import pandas as pd

from spatial_audit import (
    audit,
    audit_anndata,
    compare_methods,
    load_manifest,
    recommend_wavelengths,
)
from spatial_audit.cli import export_fixtures
from spatial_audit.fixtures import trap_cohort
from spatial_audit.report import save_report

cohort = trap_cohort("composition")
report = audit(cohort, wavelengths="auto", alpha=0.05, nugget_mode="vst")
pd.Series({key: value["status"] for key, value in report["dimensions"].items()})

## Inspect physical support

Automatic bands use the coarsest section pitch and the smallest two-dimensional section span. Each band also has an independent connected-component support check. A missing share indicates insufficient support, covariates, or estimable energy.

In [ ]:
composition = report["dimensions"]["composition"]
first = composition["sections"][0]
print(recommend_wavelengths(first["pitch_um"], first["largest_2d_span_um"]))
pd.DataFrame(composition["bands"])[["wavelength_um", "share", "supported_patients"]]

## Audit an H5AD directly

The AnnData observation table maps measurements to patients, groups and sections. The first two columns of the spatial array give calibrated positions. The direct interface labels provenance as local. This example supplies the fixture covariates explicitly.

In [ ]:
parts = []
for s in cohort.sections:
    obs = pd.DataFrame(
        {
            "patient": s.patient_id,
            "condition": s.group,
            "section_id": s.section_id,
            "batch": s.batch,
            "slide_id": s.slide_id,
        },
        index=[f"{s.section_id}-{i}" for i in range(s.X.shape[0])],
    )
    parts.append(
        ad.AnnData(
            X=s.X,
            obs=obs,
            var=pd.DataFrame(index=cohort.features),
            obsm={"spatial": s.coords, "composition": s.C},
        )
    )
adata = ad.concat(parts)
with TemporaryDirectory() as folder:
    path = Path(folder) / "example.h5ad"
    adata.write_h5ad(path)
    direct = audit_anndata(
        path,
        "patient",
        "condition",
        covariate_key="composition",
        design="stratified",
        stratum="batch",
    )
direct["dimensions"]["power"]

For an H5AD with the same column mappings, run `spatial-audit quick-run example.h5ad --patient-col patient --group-col condition --covariate-key composition --out reports`. A paired audit requires a summary in each condition for every patient. Its randomization distribution flips signs within patients.

In [ ]:
with TemporaryDirectory() as folder:
    root = Path(folder)
    export_fixtures([cohort], root)
    (loaded,) = load_manifest(root / "manifest.json")
    save_report(report, root)
    methods = root / "methods.txt"
    methods.write_text(
        f"{report['n_patients']} biological patients. No batch correction was applied.",
        encoding="utf-8",
    )
    comparison = compare_methods(methods, report)
    print(loaded.id, [row["field"] for row in comparison["matrix"]])
    print((root / "synthetic" / (loaded.id + ".md")).read_text(encoding="utf-8")[:600])